In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
from scipy.stats import norm
from dotenv import load_dotenv
from scipy.optimize import brentq
from datetime import datetime, timezone
from scipy.interpolate import RBFInterpolator
from concurrent.futures import ThreadPoolExecutor, as_completed

from api_client import TradingDeskAPI
from scanner import MarketScannerFast

In [ ]:
# Initialize all classes and parameters

load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
# load_dotenv(r"C:/Users/brian/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
DATA_DIR = os.getenv("DATA_DIR")
print(BASE_URL)

api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
scanner = MarketScannerFast(api=api)

In [ ]:
# download historical spot prices https://data.binance.vision/

start = pd.Timestamp("2026-01-01", tz="UTC")
end = pd.Timestamp("2026-08-21", tz="UTC")

def get_binance_btc_history(
    start_date="2025-01-01",
    end_date="2026-08-21",
    interval="5m"
):
    url = "https://api.binance.com/api/v3/klines"

    start_ts = pd.Timestamp(start_date).tz_convert("UTC")
    end_ts = pd.Timestamp(end_date).tz_convert("UTC")

    start_ms = int(start_ts.timestamp() * 1000)
    end_ms = int(end_ts.timestamp() * 1000)

    all_rows = []

    while start_ms < end_ms:

        params = {
            "symbol": "BTCUSDT",
            "interval": interval,
            "startTime": start_ms,
            "endTime": end_ms,
            "limit": 1000
        }

        response = requests.get(url, params=params)
        response.raise_for_status()

        rows = response.json()

        if not rows:
            break

        all_rows.extend(rows)

        # Move past last candle
        start_ms = rows[-1][0] + 1

        print(
            datetime.fromtimestamp(
                rows[-1][0] / 1000,
                tz=timezone.utc
            )
        )

        time.sleep(0.05)

    columns = [
        "timestamp",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_timestamp",
        "quote_volume",
        "num_trades",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "ignore"
    ]

    df = pd.DataFrame(all_rows, columns=columns)

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        unit="ms",
        utc=True
    )

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]

    df[numeric_columns] = df[numeric_columns].astype(float)

    df = df[
        ["timestamp", "open", "high", "low", "close", "volume"]
    ]

    return df

def download_chunk(start_date, end_date):
    print(f"Downloading {start_date} -> {end_date}")

    df = get_binance_btc_history(
        start_date=start_date,
        end_date=end_date,
        interval="5m"
    )

    print(f"Finished {start_date} -> {end_date}: {len(df)} rows")

    return df


# Create 30 chunks
dates = pd.date_range(
    start=start,
    end=end,
    periods=31
)

chunks = [
    (dates[i], dates[i + 1])
    for i in range(len(dates) - 1)
]

results = []

with ThreadPoolExecutor(max_workers=30) as executor:

    futures = {
        executor.submit(download_chunk, chunk_start, chunk_end):
        (chunk_start, chunk_end)
        for chunk_start, chunk_end in chunks
    }

    for future in as_completed(futures):

        chunk_start, chunk_end = futures[future]

        try:
            df = future.result()

            print(
                f"Finished {chunk_start} -> {chunk_end}: "
                f"{len(df):,} rows"
            )

            results.append(df)

        except Exception as e:
            print(
                f"FAILED {chunk_start} -> {chunk_end}: {e}"
            )

# Combine everything
btc = pd.concat(results, ignore_index=True)

# Sort chronologically
btc = btc.sort_values("timestamp").reset_index(drop=True)

# Remove any possible duplicate candles
btc = btc.drop_duplicates(subset=["timestamp"])

# btc.to_parquet("btc_spot_historical_price.parquet", index=False)

In [ ]:
# Download historical options data

BASE_URL = "https://www.okx.com"

def get_okx_option_instruments():

    url = f"{BASE_URL}/api/v5/public/instruments"

    params = {
        "instType": "OPTION",
        "uly": "BTC-USD"
    }

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    if data["code"] != "0":
        raise RuntimeError(data)

    return pd.DataFrame(data["data"])


instruments = get_okx_option_instruments()

print(instruments.shape)
print(instruments.columns.tolist())
print(instruments.head())

def get_okx_option_trades(inst_id, limit=100):

    url = f"{BASE_URL}/api/v5/market/history-trades"
    all_trades = []
    after = None

    while True:

        params = {
            "instId": inst_id,
            "type": "1",
            "limit": str(limit)
        }

        if after is not None:
            params["after"] = after

        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        if data["code"] != "0":
            raise RuntimeError(data)

        rows = data["data"]

        if not rows:
            break

        all_trades.extend(rows)
        new_after = rows[-1]["tradeId"]

        if new_after == after:
            break

        after = new_after

        if len(rows) < limit:
            break

        time.sleep(0.05)

    return pd.DataFrame(all_trades)

def download_option(inst_id):

    print(f"Downloading {inst_id}")

    try:

        df = get_okx_option_trades(inst_id=inst_id, limit=100)

        if not df.empty:
            df["instId"] = inst_id

        print(f"Finished {inst_id}: "f"{len(df):,} trades")

        return df

    except Exception as e:
        print(f"FAILED {inst_id}: {e}")
        return pd.DataFrame()

option_ids = instruments["instId"].tolist()

results = []

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            download_option,
            inst_id
        ): inst_id

        for inst_id in option_ids
    }

    for future in as_completed(futures):

        inst_id = futures[future]

        try:
            df = future.result()

            if not df.empty:
                results.append(df)

        except Exception as e:
            print(f"FAILED {inst_id}: {e}")

trades = (
    pd.concat(
        results,
        ignore_index=True
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

print(trades.shape)
print(trades.head())
print(trades.tail())

metadata = instruments[
    [
        "instId",
        "uly",
        "settleCcy",
        "stk",
        "optType",
        "expTime",
        "ctVal"
    ]
].copy()

options = trades.merge(
    metadata,
    on="instId",
    how="left"
)

options["timestamp"] = pd.to_datetime(
    options["ts"].astype("int64"),
    unit="ms",
    utc=True
)

options["expiry_dt"] = pd.to_datetime(
    options["expTime"].astype("int64"),
    unit="ms",
    utc=True
)

options["strike"] = pd.to_numeric(
    options["stk"],
    errors="coerce"
)

options["price"] = pd.to_numeric(
    options["px"],
    errors="coerce"
)

options["size"] = pd.to_numeric(
    options["sz"],
    errors="coerce"
)

options["T"] = (
    options["expiry_dt"] -
    options["timestamp"]
).dt.total_seconds() / (
    365.25 * 24 * 3600
)

options = options[
    options["T"] > 0
]

# options.to_parquet("eth_options.parquet")

In [ ]:
# merge options with spot price to get mark_iv

options = pd.read_parquet("eth_options.parquet")
df = options.copy()
df_non_um = df[~df["instId"].str.contains("_UM", na=False)].copy()
df_non_um

eth_spot = pd.read_parquet("eth_spot_historical_price.parquet")
options = df_non_um.copy()
spot = eth_spot.copy()

spot["timestamp"] = pd.to_datetime(spot["timestamp"], utc=True)
options["timestamp"] = pd.to_datetime(options["timestamp"], utc=True)

spot = spot.sort_values("timestamp")
options = options.sort_values("timestamp")

merged = pd.merge_asof(
    options,
    spot[["timestamp", "Open", "High", "Low", "Close", "Volume"]],
    on="timestamp",
    direction="backward"
)

merged

In [ ]:
# get mark_iv

def option_price_normalized(S, K, T, sigma, opt_type, r=0.0):
    """
    Black-Scholes price expressed in units of the underlying.

    S       : underlying/index price
    K       : strike
    T       : time to expiry in years
    sigma   : volatility
    opt_type: 'C' or 'P'
    r       : risk-free rate
    """

    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return np.nan

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

    d2 = d1 - sigma * np.sqrt(T)

    if opt_type.upper() == "C":
        # USD call price
        price_usd = (S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2))

    elif opt_type.upper() == "P":
        # USD put price
        price_usd = (K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1))

    else:
        raise ValueError("opt_type must be 'C' or 'P'")

    # Convert USD premium to underlying units
    return price_usd / S


def implied_vol(S, K, T, market_price, opt_type, r=0.0):
    """
    Calculate implied volatility for an inverse BTC/ETH option.

    market_price must be in underlying units,
    e.g. ETH for ETH-USD options.
    """

    if (
        pd.isna(S)
        or pd.isna(K)
        or pd.isna(T)
        or pd.isna(market_price)
        or S <= 0
        or K <= 0
        or T <= 0
        or market_price <= 0
    ):
        return np.nan

    opt_type = opt_type.upper()

    # No-arbitrage bounds in underlying units
    if opt_type == "C":
        lower = max(1 - (K / S) * np.exp(-r * T), 0)
        upper = 1

    elif opt_type == "P":
        lower = max((K / S) * np.exp(-r * T) - 1, 0)
        upper = (K / S) * np.exp(-r * T)

    else:
        return np.nan

    # Small tolerance for floating point issues
    eps = 1e-10

    if market_price < lower - eps or market_price > upper + eps:
        return np.nan

    def objective(sigma):
        return (option_price_normalized(S, K, T, sigma, opt_type, r) - market_price)

    try:
        return brentq(
            objective,
            1e-6,
            10.0,
            xtol=1e-8,
            rtol=1e-10,
            maxiter=100
        )

    except (ValueError, RuntimeError):
        return np.nan

copy = merged.copy()
copy["iv"] = copy.apply(
    lambda row: implied_vol(
        S=row["Close"],
        K=row["strike"],
        T=row["T"],
        market_price=row["price"],
        opt_type=row["optType"]
    ),
    axis=1
)
# copy.to_parquet("eth_options_mark_iv.parquet")
copy

In [ ]:
# polymarket btc200K contract price history
def get_price_history(token_id: str, start_ts: int, end_ts: int, interval="1m", fidelity=10):

    url = "https://clob.polymarket.com/prices-history"
    params = {
        "market": token_id,
        "startTs": start_ts,
        "endTs": end_ts,
        "interval": interval,
        "fidelity": fidelity,
    }

    r = requests.get(url, params=params, timeout=30)

    if not r.ok:
        raise RuntimeError(f"{r.status_code}: {r.text}")

    return r.json()["history"]

def get_history_chunked(token_id: str, start_ts: int, end_ts: int, chunk_days=15):
    all_history = []
    chunk_seconds = chunk_days * 24 * 60 * 60
    current = start_ts

    while current < end_ts:
        chunk_end = min(current + chunk_seconds, end_ts)

        print(datetime.fromtimestamp(current, tz=timezone.utc), "->", datetime.fromtimestamp(chunk_end, tz=timezone.utc))

        history = get_price_history(token_id, current, chunk_end,)
        all_history.extend(history)
        current = chunk_end

    # Convert to dataframe
    df = pd.DataFrame(all_history)

    if df.empty:
        return df

    df["timestamp"] = pd.to_datetime(df["t"], unit="s", utc=True)
    df["price"] = df["p"].astype(float)
    df = df[["timestamp", "price"]]

    # Remove duplicate timestamps caused by chunk boundaries
    df = (
        df
        .drop_duplicates("timestamp")
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    return df

def download_market(row, now_ts):
    idx = row.name
    token_id = row["yes_token"]

    try:
        created_at = pd.to_datetime(row["createdAt"], utc=True)
        start_ts = int(created_at.timestamp())

        history = get_history_chunked(
            token_id=token_id,
            start_ts=start_ts,
            end_ts=now_ts,
            chunk_days=15,
        )

        if history.empty:
            return idx, history, None

        history["market_index"] = idx
        history["question"] = row["question"]
        history["token_id"] = token_id

        return idx, history, None

    except Exception as e:
        return idx, None, str(e)


all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=30, liquidity_num_min=10000, volume_num_min=5000)
markets_df = scanner.scan_market(all_markets_df)
markets_df = markets_df[markets_df["question"] == "Will Bitcoin reach $200,000 by December 31, 2026?"]

now_ts = int(time.time())

all_market_history = {}
errors = {}

with ThreadPoolExecutor(max_workers=30) as executor:

    futures = {
        executor.submit(download_market, row, now_ts): row.name
        for _, row in markets_df.iterrows()
    }

    for i, future in enumerate(as_completed(futures), 1):

        idx, history, error = future.result()

        if error is not None:
            errors[idx] = error
            print(f"[{i}/{len(futures)}] ERROR market {idx}: {error}")
            continue

        all_market_history[idx] = history

        print(
            f"[{i}/{len(futures)}] "
            f"market={idx}, "
            f"observations={len(history):,}"
        )

pm_df = pd.concat(all_market_history.values(), ignore_index=True)
pm_df = pm_df.sort_values(["market_index", "timestamp"]).reset_index(drop=True)

print(pm_df.shape)
print(f"Successful: {len(all_market_history)}")
print(f"Errors: {len(errors)}")

# pm_df.to_parquet("btc200k_price.parquet")
pm_df

In [ ]:
options = pd.read_parquet("btc_options_mark_iv.parquet")
pm_df = pd.read_parquet("btc200k_price.parquet")

In [3]:
options

,instId,side,sz,px,source,tradeId,ts,uly,settleCcy,stk,...,expTime,ctVal,timestamp,expiry_dt,strike,price,size,T,spot,iv
0,BTC-USD-261225-120000-C,sell,257,0.0935,0,15,1768269960036,BTC-USD,BTC,120000,...,1798185600000,1,2026-01-13 02:06:00.036000+00:00,2026-12-25 08:00:00+00:00,120000,0.0935,257,0.947969,91290.13,0.486159
1,BTC-USD-261225-120000-C,sell,109,0.0965,0,16,1768299214032,BTC-USD,BTC,120000,...,1798185600000,1,2026-01-13 10:13:34.032000+00:00,2026-12-25 08:00:00+00:00,120000,0.0965,109,0.947042,92238.62,0.486803
2,BTC-USD-261225-160000-C,buy,238,0.0365,0,13,1768299348246,BTC-USD,BTC,160000,...,1798185600000,1,2026-01-13 10:15:48.246000+00:00,2026-12-25 08:00:00+00:00,160000,0.0365,238,0.947038,92217.77,0.484661
3,BTC-USD-261225-240000-C,sell,30,0.008,0,2,1768303125996,BTC-USD,BTC,240000,...,1798185600000,1,2026-01-13 11:18:45.996000+00:00,2026-12-25 08:00:00+00:00,240000,0.0080,30,0.946918,92149.42,0.509809
4,BTC-USD-261225-160000-C,sell,10,0.037,0,14,1768308858706,BTC-USD,BTC,160000,...,1798185600000,1,2026-01-13 12:54:18.706000+00:00,2026-12-25 08:00:00+00:00,160000,0.0370,10,0.946737,92122.57,0.487295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100906,BTC-USD-260823-72750-P,sell,150,0.009,0,1,1787270527848,BTC-USD,BTC,72750,...,1787472000000,1,2026-08-21 00:02:07.848000+00:00,2026-08-23 08:00:00+00:00,72750,0.0090,150,0.006384,73107.36,0.354619
100907,BTC-USD-260822-76000-C,sell,100,0.0013,0,49,1787270604277,BTC-USD,BTC,76000,...,1787385600000,1,2026-08-21 00:03:24.277000+00:00,2026-08-22 08:00:00+00:00,76000,0.0013,100,0.003644,73107.36,0.486255
100908,BTC-USD-260925-95000-C,sell,1,0.0023,0,1092,1787270624154,BTC-USD,BTC,95000,...,1790323200000,1,2026-08-21 00:03:44.154000+00:00,2026-09-25 08:00:00+00:00,95000,0.0023,1,0.096730,73107.36,0.465619
100909,BTC-USD-260822-75000-C,sell,72,0.0026,0,39,1787270682538,BTC-USD,BTC,75000,...,1787385600000,1,2026-08-21 00:04:42.538000+00:00,2026-08-22 08:00:00+00:00,75000,0.0026,72,0.003642,73107.36,0.452573


In [7]:
pm_df

,timestamp,price
0,2026-01-11 06:00:22+00:00,0.0850
1,2026-01-11 06:10:16+00:00,0.0850
2,2026-01-11 06:20:14+00:00,0.0850
3,2026-01-11 06:30:17+00:00,0.0850
4,2026-01-11 06:40:17+00:00,0.0850
...,...,...
30852,2026-08-23 05:40:14+00:00,0.0205
30853,2026-08-23 05:50:15+00:00,0.0205
30854,2026-08-23 05:54:13+00:00,0.0205
30855,2026-08-23 05:55:13+00:00,0.0205


In [ ]:
# ============================================================
# CONFIG
# ============================================================
TARGET_STRIKE = 200000

# Polymarket assumed round-trip spread
ASSUMED_SPREAD = 0.01

# Example fee.
# Replace with your actual Polymarket fee calculation.
fee_rate = 0.07

def fee(price, fee_rate):
    #fee = C × feeRate × p × (1 - p)
    #Where C = number of shares traded and p = price of the shares.
    return fee_rate * price * (1 - price)

# How close an options observation must be to the PM timestamp.
MAX_OPTIONS_STALENESS = pd.Timedelta(minutes=10)

# Forward horizons to test
HORIZONS = {
    "1h": 60 * 60 * 1000,
    "4h": 4 * 60 * 60 * 1000,
    "24h": 24 * 60 * 60 * 1000,
    "72h": 3 * 24 * 60 * 60 * 1000,
    "1w": 7 * 24 * 60 * 60 * 1000,
}

PM_EXPIRY = pd.Timestamp(
    "2026-12-31 23:59:59",
    tz="UTC"
)

OPTION_EXPIRY = pd.Timestamp(
    "2026-12-25 08:00:00",
    tz="UTC"
)

# ============================================================
# MODEL
# ============================================================
class OptionsSurface:
    # --------------------------------------------------------
    # Build variance surface
    # --------------------------------------------------------

    def build_variance_surface(self, weighted_spot, surface_df):

        surface_df = surface_df.copy()

        K = surface_df["strike"].to_numpy(dtype=float)
        T = surface_df["T"].to_numpy(dtype=float)

        # Remove invalid observations
        valid = (
            np.isfinite(K)
            & np.isfinite(T)
            & (K > 0)
            & (T > 0)
            & np.isfinite(surface_df["total_variance"])
            & (surface_df["total_variance"] > 0)
        )

        surface_df = surface_df.loc[valid].copy()

        K = surface_df["strike"].to_numpy(dtype=float)
        T = surface_df["T"].to_numpy(dtype=float)

        # Log-moneyness
        k = np.log(K / weighted_spot)

        # Avoid zero std
        k_std = k.std()
        T_std = T.std()

        if k_std == 0 or T_std == 0:
            raise ValueError("Not enough strike/maturity variation to build surface.")

        k_mean = k.mean()
        T_mean = T.mean()

        k_scaled = (k - k_mean) / k_std
        T_scaled = (T - T_mean) / T_std

        query_points = np.column_stack([k_scaled, T_scaled])

        # Interpolate log(total variance)
        log_variance = np.log(surface_df["total_variance"].to_numpy(dtype=float))

        f_variance = RBFInterpolator(
            query_points,
            log_variance,
            kernel="thin_plate_spline",
            smoothing=0.001
        )

        params = {
            "k_mean": k_mean,
            "k_std": k_std,
            "T_mean": T_mean,
            "T_std": T_std,
        }

        return f_variance, params


    # --------------------------------------------------------
    # Get IV from variance surface
    # --------------------------------------------------------

    def get_iv_from_surface(self, weighted_spot, f_variance, params, required_strike, T):

        k_mean = params["k_mean"]
        k_std = params["k_std"]
        T_mean = params["T_mean"]
        T_std = params["T_std"]

        if T <= 0:
            return np.nan

        if k_std == 0 or T_std == 0:
            return np.nan

        k = np.log(required_strike / weighted_spot)

        k_scaled = (k - k_mean) / k_std
        T_scaled = (T - T_mean) / T_std

        query_points = np.array([[k_scaled, T_scaled]])

        log_variance = f_variance(query_points)[0]
        total_variance = np.exp(log_variance)

        iv = np.sqrt(total_variance / T)

        return float(iv)


    # --------------------------------------------------------
    # Probability BTC touches above strike
    # --------------------------------------------------------

    def prob_touch_above(self, spot, required_strike, iv, T, r=0.0, q=0.0):

        if(spot <= 0 or required_strike <= 0 or iv <= 0 or T <= 0):
            return np.nan

        mu = r - q - 0.5 * (iv ** 2)

        sqrt_T = np.sqrt(T)

        log_ratio = np.log(required_strike / spot)

        d1 = (mu * T - log_ratio) / (iv * sqrt_T)

        d2 = (-mu * T - log_ratio) / (iv * sqrt_T)

        exponent = (2 * mu / iv ** 2)

        p_touch = (norm.cdf(d1) + (required_strike / spot) ** exponent * norm.cdf(d2))

        return float(np.clip(p_touch, 0.0, 1.0))


# ============================================================
# OPTIONS DATA PREPARATION
# ============================================================

def prepare_options_data(options_df, timestamp_col="timestamp", expiry_col="expiry_dt", strike_col="strike", iv_col="iv", spot_col="spot"):
    """
    Convert raw options data into the fields required
    by the variance surface.

    Assumes IV is decimal:
        0.50 = 50%

    """

    df = options_df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], utc=True)
    df[expiry_col] = pd.to_datetime(df[expiry_col], utc=True)
    df[strike_col] = pd.to_numeric(df[strike_col], errors="coerce")
    df[iv_col] = pd.to_numeric(df[iv_col], errors="coerce")
    df[spot_col] = pd.to_numeric(df[spot_col], errors="coerce")

    # Time to expiry in years
    df["T"] = ((df[expiry_col] - df[timestamp_col]).dt.total_seconds() / (365.25 * 24 * 3600))

    # Total variance
    df["total_variance"] = (df[iv_col] ** 2 * df["T"])

    # Standardize column names
    df = df.rename(columns={
        timestamp_col: "timestamp",
        expiry_col: "expiry_dt",
        strike_col: "strike",
        spot_col: "spot",
    })

    df = df[
        [
            "timestamp",
            "expiry_dt",
            "strike",
            "spot",
            "iv",
            "T",
            "total_variance",
        ]
    ]

    # Remove invalid observations
    df = df[
        np.isfinite(df["strike"])
        & np.isfinite(df["spot"])
        & np.isfinite(df["iv"])
        & np.isfinite(df["T"])
        & (df["strike"] > 0)
        & (df["spot"] > 0)
        & (df["iv"] > 0)
        & (df["T"] > 0)
    ]

    return df


# ============================================================
# GET OPTIONS SURFACE FOR A PARTICULAR PM TIMESTAMP
# ============================================================
def get_surface_at_timestamp(
    options_df,
    pm_timestamp,
    lookback_minutes=60,
):
    pm_timestamp = pd.Timestamp(pm_timestamp)

    start = pm_timestamp - pd.Timedelta(
        minutes=lookback_minutes
    )

    available = options_df[
        (options_df["timestamp"] >= start) &
        (options_df["timestamp"] <= pm_timestamp) &
        (options_df["expiry_dt"] == OPTION_EXPIRY)
    ].copy()

    if available.empty:
        return None

    # Latest observation for each strike
    surface = (
        available
        .sort_values("timestamp")
        .groupby("strike", as_index=False)
        .tail(1)
        .copy()
    )

    # Age of each option observation
    surface["age_minutes"] = (
        pm_timestamp - surface["timestamp"]
    ).dt.total_seconds() / 60

    return surface

# ============================================================
# BUILD MODEL PROBABILITY FOR ONE PM OBSERVATION
# ============================================================

def get_target_iv(
    model,
    expiry_surface,
    spot,
    target_strike,
    T
    ):
        if expiry_surface is None or expiry_surface.empty:
            return np.nan, "no_surface"

        # ------------------------------------------------
        # 1. Direct observation exists
        # ------------------------------------------------

        exact = expiry_surface[
            np.isclose(
                expiry_surface["strike"],
                target_strike,
                rtol=0,
                atol=1.0,
            )
        ]

        if not exact.empty:

            obs = (
                exact
                .sort_values("timestamp")
                .iloc[-1]
            )

            iv = float(obs["iv"])

            if np.isfinite(iv) and 0 < iv < 5:
                return iv, "observed"

            return np.nan, "invalid_observed_iv"

        # ------------------------------------------------
        # 2. Need interpolation
        # ------------------------------------------------

        strikes = np.sort(
            expiry_surface["strike"]
            .dropna()
            .unique()
        )

        if len(strikes) < 3:
            return np.nan, "not_enough_strikes"

        # Do NOT extrapolate
        if (
            target_strike < strikes.min()
            or target_strike > strikes.max()
        ):
            return np.nan, "outside_range"

        # ------------------------------------------------
        # 3. Interpolate
        # ------------------------------------------------

        try:

            f_variance, params = (
                model.build_variance_surface(
                    weighted_spot=spot,
                    surface_df=expiry_surface,
                )
            )

            iv = model.get_iv_from_surface(
                weighted_spot=spot,
                f_variance=f_variance,
                params=params,
                required_strike=target_strike,
                T=T,
            )

        except Exception:
            return np.nan, "interpolation_failed"

        if not np.isfinite(iv):
            return np.nan, "invalid_interpolated_iv"

        if iv <= 0 or iv > 5:
            return np.nan, "implausible_interpolated_iv"

        return float(iv), "interpolated"


def calculate_model_probability(model, surface, pm_timestamp, target_strike=TARGET_STRIKE, risk_free_rate=0.0, dividend_yield=0.0):
    """
    Calculate probability that BTC touches target_strike
    before the relevant expiry.

    IMPORTANT:
    This implementation assumes the options surface contains
    an expiry corresponding to the Polymarket event expiry.
    """

    if surface is None or surface.empty:
        return None

    # Use latest available spot from surface
    spot = surface["spot"].median()

    if not np.isfinite(spot) or spot <= 0:
        return None

    # --------------------------------------------------------
    # IMPORTANT:
    # Pick the relevant expiry.
    #
    # Ideally this should be the option expiry matching
    # the Polymarket resolution date.
    #
    # Here we choose the latest available expiry.
    # Change this if you have an exact expiry.
    # --------------------------------------------------------
    T = (PM_EXPIRY - pd.Timestamp(pm_timestamp)).total_seconds() / (365.25 * 24 * 3600)

    if T <= 0:
        return None

    # --------------------------------------------------------
    # Build variance surface
    # --------------------------------------------------------
    try:
        iv, iv_source = get_target_iv(model, surface, spot, target_strike, T)
    except:
        return None

    if not np.isfinite(iv):
        return None

    # --------------------------------------------------------
    # Calculate touch probability
    # --------------------------------------------------------
    p_touch = model.prob_touch_above(
        spot=spot,
        required_strike=target_strike,
        iv=iv,
        T=T,
        r=risk_free_rate,
        q=dividend_yield,
    )

    return {
        "spot": spot,
        "expiry": PM_EXPIRY,
        "T": T,
        "iv_target": iv,
        "model_prob": p_touch,
        "options_timestamp": surface["timestamp"].max(),
    }


# ============================================================
# FUTURE PM PRICE
# ============================================================

def add_future_prices(df, horizons=HORIZONS):
    """
    Find the next available Polymarket price around
    each future horizon.
    """

    df = df.sort_values("timestamp").reset_index(drop=True)

    base = df[["timestamp", "pm_price"]].copy()

    for name, horizon in horizons.items():

        future = base.copy()

        future["target_timestamp"] = (future["timestamp"] + pd.to_timedelta(horizon, unit="ms"))

        future = future.rename(columns={"pm_price": f"pm_{name}"})

        result = pd.merge_asof(df[["timestamp"]].sort_values("timestamp"),

            future[["target_timestamp", f"pm_{name}"]].sort_values("target_timestamp"),

            left_on="timestamp",
            right_on="target_timestamp",

            direction="forward",

            tolerance=pd.Timedelta(minutes=5),
        )

        df[f"pm_{name}"] = result[f"pm_{name}"].values

    return df


# ============================================================
# CONVERGENCE CALCULATIONS
# ============================================================

def calculate_convergence(df, horizons=HORIZONS):
    """
    Anchor convergence to the model probability
    at ENTRY.

    This prevents later model revisions from creating
    fake convergence.
    """

    for name in horizons:

        model = df["model_prob"]
        current = df["pm_price"]
        future = df[f"pm_{name}"]

        initial_gap = (model - current)

        future_gap = (model - future)

        df[f"gap_{name}"] = future_gap

        df[f"convergence_{name}"] = (initial_gap.abs() - future_gap.abs())

        # Positive means PM moved in the direction
        # of the model.
        df[f"directional_move_{name}"] = (np.sign(initial_gap) * (future - current))

    return df


# ============================================================
# SYNTHETIC EXECUTION
# ============================================================

def backtest_ev_strategy(df, spread=ASSUMED_SPREAD, fee_rate=0.07, ENTRY_EV_THRESHOLD=0.02, EXIT_EV_THRESHOLD=0.02):
    """
    Stateful EV strategy.

    LONG YES:
        Enter when EV > 0.
        Hold while EV > 0.
        Exit when EV <= 0.

    Uses synthetic bid/ask:
        ask = pm_price + spread / 2
        bid = pm_price - spread / 2

    Fees:
        fee(price) = fee_rate * price * (1 - price)
    """

    df = df.sort_values("timestamp").reset_index(drop=True).copy()

    half_spread = spread / 2

    # --------------------------------------------------------
    # Synthetic executable prices
    # --------------------------------------------------------

    df["ask"] = df["pm_price"] + half_spread
    df["bid"] = df["pm_price"] - half_spread

    # Keep prices in [0, 1]
    df["ask"] = df["ask"].clip(0, 1)
    df["bid"] = df["bid"].clip(0, 1)

    # --------------------------------------------------------
    # Entry EV
    # --------------------------------------------------------

    df["entry_fee"] = fee(df["ask"], fee_rate)

    df["entry_ev"] = (df["model_prob"] - df["ask"] - df["entry_fee"])

    # --------------------------------------------------------
    # Strategy state
    # --------------------------------------------------------

    in_position = False

    entry_price = np.nan
    entry_fee = np.nan
    entry_timestamp = pd.NaT
    entry_model_prob = np.nan

    results = []

    for i, row in df.iterrows():

        timestamp = row["timestamp"]
        model_prob = row["model_prob"]
        ask = row["ask"]
        bid = row["bid"]

        # Skip rows where model unavailable
        if not np.isfinite(model_prob):
            results.append({
                "position": int(in_position),
                "action": None,
                "entry_timestamp": entry_timestamp,
                "entry_price": entry_price,
                "exit_price": np.nan,
                "trade_pnl": np.nan,
            })
            continue

        # ====================================================
        # NOT IN POSITION
        # ====================================================

        if not in_position:

            entry_fee_current = fee(ask, fee_rate)

            entry_ev = (model_prob - ask - entry_fee_current)

            # ENTER
            if entry_ev > ENTRY_EV_THRESHOLD:

                in_position = True

                entry_price = ask
                entry_fee = entry_fee_current
                entry_timestamp = timestamp
                entry_model_prob = model_prob

                results.append({
                    "position": 1,
                    "action": "ENTER",
                    "entry_timestamp": entry_timestamp,
                    "entry_price": entry_price,
                    "entry_model_prob": entry_model_prob,
                    "exit_price": np.nan,
                    "trade_pnl": np.nan,
                })

            else:

                results.append({
                    "position": 0,
                    "action": None,
                    "entry_timestamp": pd.NaT,
                    "entry_price": np.nan,
                    "entry_model_prob": np.nan,
                    "exit_price": np.nan,
                    "trade_pnl": np.nan,
                })

        # ====================================================
        # ALREADY IN POSITION
        # ====================================================

        else:
            # ------------------------------------------------
            # EXIT
            # -----------------------------------------
            exit_fee_current = fee(bid, fee_rate)

            # EV if we were to exit / current model value
            exit_ev = (bid - exit_fee_current)

            # ------------------------------------------------
            # HOLD
            # ------------------------------------------------
            expected_exit_price = model_prob - half_spread
            expected_exit_fee = fee(expected_exit_price, fee_rate)

            hold_ev = expected_exit_price - expected_exit_fee

            if (hold_ev - exit_ev) <= EXIT_EV_THRESHOLD:

                trade_pnl = (bid - entry_price - entry_fee - exit_fee_current)

                results.append({
                    "position": 0,
                    "action": "EXIT",
                    "entry_timestamp": entry_timestamp,
                    "entry_price": entry_price,
                    "entry_model_prob": entry_model_prob,
                    "exit_price": bid,
                    "exit_model_prob": model_prob,
                    "exit_timestamp": timestamp,
                    "trade_pnl": trade_pnl,
                })

                # Reset
                in_position = False

                entry_price = np.nan
                entry_fee = np.nan
                entry_timestamp = pd.NaT
                entry_model_prob = np.nan

            # ------------------------------------------------
            # HOLD
            # ------------------------------------------------
            else:
                results.append({
                    "position": 1,
                    "action": "HOLD",
                    "entry_timestamp": entry_timestamp,
                    "entry_price": entry_price,
                    "entry_model_prob": entry_model_prob,
                    "exit_price": np.nan,
                    "exit_model_prob": np.nan,
                    "exit_timestamp": pd.NaT,
                    "trade_pnl": np.nan,
                })

    result_df = pd.DataFrame(results)

    return pd.concat([df.reset_index(drop=True), result_df], axis=1)

def calculate_convergence_ratios(df, horizons=HORIZONS):

    df = df.copy()

    edge = df["model_prob"] - df["pm_price"]

    for name in horizons:

        future = df[f"pm_{name}"]

        # How much of the original model-vs-market gap
        # has been closed?
        df[f"convergence_ratio_{name}"] = np.where(
            edge.abs() > 1e-8,
            (future - df["pm_price"]) / edge,
            np.nan
        )

    return df

# ============================================================
# MAIN BACKTEST
# ============================================================

def build_backtest(polymarket_df, options_df):
    """
    Main pipeline.
    """

    pm = polymarket_df.copy()

    # --------------------------------------------------------
    # Clean PM data
    # --------------------------------------------------------
    pm["timestamp"] = pd.to_datetime(pm["timestamp"], utc=True)
    pm["pm_price"] = pd.to_numeric(pm["price"], errors="coerce")
    pm = pm[["timestamp", "pm_price"]].dropna()
    pm = pm.sort_values("timestamp").reset_index(drop=True)

    # --------------------------------------------------------
    # Prepare options
    # --------------------------------------------------------
    options = prepare_options_data(options_df)
    options = options.sort_values("timestamp").reset_index(drop=True)

    # # --------------------------------------------------------
    # # Calculate model probability for every PM timestamp
    # # --------------------------------------------------------

    model = OptionsSurface()

    results = []

    for i, row in pm.iterrows():

        timestamp = row["timestamp"]
        print(timestamp)

        surface = get_surface_at_timestamp(options, timestamp)

        result = calculate_model_probability(
            model=model,
            surface=surface,
            pm_timestamp=timestamp,
            target_strike=TARGET_STRIKE,
        )

        if result is None:
            results.append({
                "timestamp": timestamp,
                "pm_price": row["pm_price"],
                "model_prob": np.nan,
                "spot": np.nan,
                "T": np.nan,
                "iv_target": np.nan,
                "options_timestamp": pd.NaT,
                "expiry": pd.NaT,
            })

        else:

            results.append({
                "timestamp": timestamp,
                "pm_price": row["pm_price"],
                **result,
            })

    df = pd.DataFrame(results)

    # # --------------------------------------------------------
    # # Remove rows where model couldn't be calculated
    # # --------------------------------------------------------
    df = df.dropna(subset=["model_prob"]).reset_index(drop=True)

    # # --------------------------------------------------------
    # # Edge
    # # --------------------------------------------------------
    df["edge"] = (df["model_prob"] - df["pm_price"])

    # --------------------------------------------------------
    # Future PM prices
    # --------------------------------------------------------
    df = add_future_prices(df)

    # # --------------------------------------------------------
    # # Convergence
    # # --------------------------------------------------------
    df = calculate_convergence(df)
    df = calculate_convergence_ratios(df)

    # # --------------------------------------------------------
    # # Synthetic execution
    # # --------------------------------------------------------
    df = backtest_ev_strategy(df, spread=ASSUMED_SPREAD, fee_rate=0.07, ENTRY_EV_THRESHOLD=0.02, EXIT_EV_THRESHOLD=0.02)

    return df, options

# ============================================================
# LOAD YOUR DATA
# ============================================================
pm_df = pd.read_parquet("btc200k_price.parquet")
pm_df = pm_df[pm_df["timestamp"] >= "2026-01-13"]
options_df = pd.read_parquet("btc_options_mark_iv.parquet")

# ============================================================
# RUN
# ============================================================
backtest, options = build_backtest(polymarket_df=pm_df, options_df=options_df)

# ============================================================
# SAVE
# ============================================================
backtest.to_parquet("btc_200k_backtest.parquet", index=False)

backtest

2026-01-13 00:00:27+00:00
2026-01-13 00:10:15+00:00
2026-01-13 00:20:17+00:00
2026-01-13 00:30:20+00:00
2026-01-13 00:40:25+00:00
2026-01-13 00:50:14+00:00
2026-01-13 01:10:16+00:00
2026-01-13 01:20:16+00:00
2026-01-13 01:30:19+00:00
2026-01-13 01:40:31+00:00
2026-01-13 01:50:16+00:00
2026-01-13 02:00:19+00:00
2026-01-13 02:10:30+00:00
2026-01-13 02:20:30+00:00
2026-01-13 02:30:21+00:00
2026-01-13 02:40:16+00:00
2026-01-13 02:50:15+00:00
2026-01-13 03:00:26+00:00
2026-01-13 03:10:16+00:00
2026-01-13 03:20:14+00:00
2026-01-13 03:30:34+00:00
2026-01-13 03:40:15+00:00
2026-01-13 03:50:16+00:00
2026-01-13 04:00:20+00:00
2026-01-13 04:10:16+00:00
2026-01-13 04:20:16+00:00
2026-01-13 04:30:20+00:00
2026-01-13 04:40:17+00:00
2026-01-13 04:50:18+00:00
2026-01-13 05:00:20+00:00
2026-01-13 05:10:16+00:00
2026-01-13 05:20:38+00:00
2026-01-13 05:30:20+00:00
2026-01-13 05:40:15+00:00
2026-01-13 05:50:38+00:00
2026-01-13 06:00:26+00:00
2026-01-13 06:10:15+00:00
2026-01-13 06:20:17+00:00
2026-01-13 0

C:\Users\brian\AppData\Local\Temp\ipykernel_16600\2770639941.py:150: RuntimeWarning: overflow encountered in exp
  total_variance = np.exp(log_variance)
C:\Users\brian\AppData\Local\Temp\ipykernel_16600\2770639941.py:150: RuntimeWarning: overflow encountered in exp
  total_variance = np.exp(log_variance)


2026-03-29 00:20:43+00:00
2026-03-29 00:30:53+00:00
2026-03-29 00:40:42+00:00
2026-03-29 00:50:43+00:00
2026-03-29 01:00:55+00:00
2026-03-29 01:10:46+00:00
2026-03-29 01:20:37+00:00
2026-03-29 01:30:42+00:00
2026-03-29 01:40:37+00:00
2026-03-29 01:50:41+00:00
2026-03-29 02:00:55+00:00
2026-03-29 02:10:39+00:00
2026-03-29 02:20:37+00:00
2026-03-29 02:40:45+00:00
2026-03-29 02:50:39+00:00
2026-03-29 03:00:55+00:00
2026-03-29 03:10:41+00:00
2026-03-29 03:20:52+00:00
2026-03-29 03:40:55+00:00
2026-03-29 03:50:54+00:00
2026-03-29 04:00:47+00:00
2026-03-29 04:10:40+00:00
2026-03-29 04:20:37+00:00
2026-03-29 04:30:47+00:00
2026-03-29 04:40:37+00:00
2026-03-29 04:50:56+00:00
2026-03-29 05:00:55+00:00
2026-03-29 05:10:46+00:00
2026-03-29 05:20:43+00:00
2026-03-29 05:40:38+00:00
2026-03-29 05:50:41+00:00
2026-03-29 06:10:40+00:00
2026-03-29 06:20:42+00:00
2026-03-29 06:40:38+00:00
2026-03-29 06:50:38+00:00
2026-03-29 07:00:52+00:00
2026-03-29 07:10:41+00:00
2026-03-29 07:20:39+00:00
2026-03-29 0

C:\Users\brian\AppData\Local\Temp\ipykernel_16600\2770639941.py:150: RuntimeWarning: overflow encountered in exp
  total_variance = np.exp(log_variance)


2026-05-25 13:50:04+00:00
2026-05-25 14:00:05+00:00
2026-05-25 14:10:04+00:00
2026-05-25 14:20:05+00:00
2026-05-25 14:30:05+00:00
2026-05-25 14:40:16+00:00
2026-05-25 14:50:06+00:00
2026-05-25 15:00:08+00:00
2026-05-25 15:10:19+00:00
2026-05-25 15:20:04+00:00
2026-05-25 15:30:04+00:00
2026-05-25 15:40:05+00:00
2026-05-25 15:50:04+00:00
2026-05-25 16:00:07+00:00
2026-05-25 16:10:07+00:00
2026-05-25 16:20:07+00:00
2026-05-25 16:30:06+00:00
2026-05-25 16:40:04+00:00
2026-05-25 16:50:05+00:00
2026-05-25 17:00:10+00:00
2026-05-25 17:10:04+00:00
2026-05-25 17:20:04+00:00
2026-05-25 17:30:05+00:00
2026-05-25 17:40:04+00:00
2026-05-25 17:50:04+00:00
2026-05-25 18:00:08+00:00
2026-05-25 18:10:04+00:00
2026-05-25 18:20:04+00:00
2026-05-25 18:30:05+00:00
2026-05-25 18:40:05+00:00
2026-05-25 18:50:04+00:00
2026-05-25 19:00:04+00:00
2026-05-25 19:10:06+00:00
2026-05-25 19:20:21+00:00
2026-05-25 19:30:06+00:00
2026-05-25 19:40:05+00:00
2026-05-25 19:50:06+00:00
2026-05-25 20:00:08+00:00
2026-05-25 2

,timestamp,pm_price,model_prob,spot,T,iv_target,options_timestamp,expiry,edge,pm_1h,...,entry_ev,position,action,entry_timestamp,entry_price,entry_model_prob,exit_price,trade_pnl,exit_model_prob,exit_timestamp
0,2026-01-14 23:00:24+00:00,0.0950,0.093989,96878.34,0.961099,0.498104,2026-01-14 22:57:41.393000+00:00,2026-12-31 23:59:59+00:00,-0.001011,NaN,...,-0.012311,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT
1,2026-01-14 23:10:22+00:00,0.0950,0.095520,97231.51,0.961080,0.498104,2026-01-14 23:06:48.038000+00:00,2026-12-31 23:59:59+00:00,0.000520,NaN,...,-0.010780,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT
2,2026-01-14 23:20:18+00:00,0.0950,0.095518,97231.51,0.961061,0.498104,2026-01-14 23:06:48.038000+00:00,2026-12-31 23:59:59+00:00,0.000518,NaN,...,-0.010782,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT
3,2026-01-14 23:30:24+00:00,0.0950,0.095515,97231.51,0.961042,0.498104,2026-01-14 23:06:48.038000+00:00,2026-12-31 23:59:59+00:00,0.000515,NaN,...,-0.010785,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT
4,2026-01-14 23:40:17+00:00,0.0950,0.095512,97231.51,0.961023,0.498104,2026-01-14 23:06:48.038000+00:00,2026-12-31 23:59:59+00:00,0.000512,NaN,...,-0.010788,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,2026-07-26 14:10:08+00:00,0.0225,0.003293,64686.58,0.433702,0.622430,2026-07-26 14:07:35.626000+00:00,2026-12-31 23:59:59+00:00,-0.019207,NaN,...,-0.026079,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT
147,2026-07-26 14:20:07+00:00,0.0225,0.003293,64686.58,0.433683,0.622430,2026-07-26 14:07:35.626000+00:00,2026-12-31 23:59:59+00:00,-0.019207,NaN,...,-0.026079,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT
148,2026-07-26 14:30:32+00:00,0.0225,0.003292,64686.58,0.433663,0.622430,2026-07-26 14:07:35.626000+00:00,2026-12-31 23:59:59+00:00,-0.019208,NaN,...,-0.026080,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT
149,2026-07-26 14:40:07+00:00,0.0225,0.003291,64686.58,0.433645,0.622430,2026-07-26 14:07:35.626000+00:00,2026-12-31 23:59:59+00:00,-0.019209,NaN,...,-0.026081,0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaT


In [67]:
t = backtest.dropna(subset=["trade_pnl"])
t

,timestamp,pm_price,model_prob,spot,T,iv_target,options_timestamp,expiry,edge,pm_1h,...,entry_ev,position,action,entry_timestamp,entry_price,entry_model_prob,exit_price,trade_pnl,exit_model_prob,exit_timestamp
54,2026-03-17 03:00:47+00:00,0.0505,0.028479,75219.135,0.793633,0.553897,2026-03-17 02:56:34.182000+00:00,2026-12-31 23:59:59+00:00,-0.022021,NaN,...,-0.03069,0,EXIT,2026-03-17 02:00:42+00:00,0.0555,0.157803,0.0455,-0.016709,0.028479,2026-03-17 03:00:47+00:00


In [54]:
research = backtest.copy()

research["edge"] = (
    research["model_prob"]
    - research["pm_price"]
)

research["edge_bucket"] = pd.qcut(
    research["edge"],
    10,
    duplicates="drop"
)

for horizon in ["1h", "4h", "24h", "72h", "1w"]:

    result = (
        research
        .groupby("edge_bucket", observed=True)
        .agg(
            mean_edge=("edge", "mean"),
            mean_move=(
                f"pm_{horizon}",
                lambda x: np.nanmean(
                    x - research.loc[x.index, "pm_price"]
                )
            ),
            n=("edge", "size")
        )
    )

    print("\n", horizon)
    print(result)



 1h
                    mean_edge  mean_move     n
edge_bucket                                   
(-0.1006, -0.0491]  -0.059658  -0.000003  2220
(-0.0491, -0.0423]  -0.045627  -0.000018  2219
(-0.0423, -0.0384]  -0.040584  -0.000008  2219
(-0.0384, -0.0331]  -0.035875  -0.000020  2219
(-0.0331, -0.0224]  -0.025942   0.000024  2219
(-0.0224, -0.0215]  -0.021764  -0.000007  2219
(-0.0215, -0.0205]  -0.020775   0.000005  2219
(-0.0205, -0.0185]  -0.019469  -0.000014  2219
(-0.0185, -0.0155]  -0.016965   0.000025  2219
(-0.0155, 0.399]     0.066000   0.000052  2219

 4h
                    mean_edge  mean_move     n
edge_bucket                                   
(-0.1006, -0.0491]  -0.059658  -0.000160  2220
(-0.0491, -0.0423]  -0.045627   0.000072  2219
(-0.0423, -0.0384]  -0.040584  -0.000072  2219
(-0.0384, -0.0331]  -0.035875   0.000046  2219
(-0.0331, -0.0224]  -0.025942   0.000189  2219
(-0.0224, -0.0215]  -0.021764  -0.000031  2219
(-0.0215, -0.0205]  -0.020775  -0.000008  2219
(-0

In [55]:
print(
    backtest[
        ["model_prob", "pm_price", "edge", "iv_target", "spot", "T"]
    ].describe(percentiles=[
        0.01, 0.05, 0.10, 0.25,
        0.50, 0.75, 0.90, 0.95, 0.99
    ])
)

         model_prob      pm_price          edge      iv_target          spot  \
count  2.219100e+04  22191.000000  22191.000000   2.219100e+04  22191.000000   
mean   1.452092e-02      0.036588     -0.022068  3.957138e+149  69251.301305   
std    5.421739e-02      0.020419      0.051813  4.166293e+151   7446.213728   
min    0.000000e+00      0.014500     -0.099623  2.937838e-156  58290.170000   
1%     0.000000e+00      0.014500     -0.073967   7.951815e-12  59364.875000   
5%     6.526652e-41      0.015500     -0.056741   1.245645e-01  60825.317500   
10%    2.659046e-17      0.018000     -0.049056   2.007591e-01  62472.300000   
25%    2.232285e-07      0.020500     -0.040731   3.075494e-01  63742.220000   
50%    4.234390e-04      0.023000     -0.022423   4.092184e-01  66117.240000   
75%    6.088943e-03      0.048500     -0.019500   4.875573e-01  75002.450000   
90%    2.311673e-02      0.065000     -0.015500   5.528825e-01  79274.910000   
95%    5.510898e-02      0.085000     -0

In [ ]:
df = backtest.copy()

for name in ["1h", "4h", "24h", "72h", "1w"]:
    df[f"move_{name}"] = df[f"pm_{name}"] - df["pm_price"]


print(
    df[
        [
            "edge",
            "move_1h",
            "move_4h",
            "move_24h",
            "move_72h",
            "move_1w"
        ]
    ].describe()
)

# “My second hypothesis was that exchange-option information could identify persistent probability dislocations in crypto prediction markets. 
# I mapped the option surface into model-implied touch probabilities and compared those estimates with executable prediction-market prices after fees.”

# “I initially expected sufficiently large dislocations to converge toward the model-implied probability over short horizons. 
# Instead, I found that the model-market differences were persistent, with little evidence of short-horizon convergence.”

# “This changed the research question. 
# Rather than assuming that persistent disagreement represents a prediction-market lag, 
# I am now testing whether the disagreement reflects probability-model misspecification, 
# the risk-neutral-to-physical measure transformation, the mapping from vanilla option surfaces to path-dependent touch probabilities, 
# differences in risk premia, or prediction-market liquidity and execution constraints.”

# That's more defensible because you're no longer assuming which market is right.


               edge       move_1h       move_4h      move_24h     move_72h  \
count  22191.000000  11778.000000  10675.000000  10257.000000  9530.000000   
mean      -0.022068      0.000004      0.000021      0.000311     0.001211   
std        0.051813      0.000705      0.001221      0.002671     0.004182   
min       -0.099623     -0.010000     -0.018500     -0.017000    -0.021500   
25%       -0.040731      0.000000      0.000000      0.000000     0.000000   
50%       -0.022423      0.000000      0.000000      0.000000     0.000500   
75%       -0.019500      0.000000      0.000000      0.001000     0.002000   
max        0.399359      0.010000      0.015000      0.011500     0.030000   

           move_1w  
count  8696.000000  
mean      0.002717  
std       0.005492  
min      -0.020500  
25%      -0.000500  
50%       0.002000  
75%       0.006000  
max       0.020000  
